In [0]:
from pyspark.sql import functions as F

In [0]:
base_vol_path = "/Volumes/databricksbootcamp/northwindcasestudy/northwindcasestudyvolume"

In [0]:
(
    spark.read
    .csv(f"{base_vol_path}/vends.csv", header=True, inferSchema=True)
    .write.mode("overwrite").saveAsTable("databricksbootcamp.northwindcasestudy.vends")
)

(
    spark.read
    .csv(f"{base_vol_path}/machines.csv", header=True, inferSchema=True)
    .write.mode("overwrite").saveAsTable("databricksbootcamp.northwindcasestudy.machines")
)

In [0]:
vends = spark.read.table("databricksbootcamp.northwindcasestudy.vends").filter(F.col("machine_id") == "M1004")
machines = spark.read.table("databricksbootcamp.northwindcasestudy.machines")

In [0]:
q = vends.filter(F.col("machine_id") == "M1004").groupBy("machine_id").sum("amount")
q.explain()

In [0]:
q.explain(True)

In [0]:
q.explain("formatted")

In [0]:
chain = vends.filter(F.col("machine_id") == "M1004").select("vend_date", "amount")
chain.count()

In [0]:
vends.filter(F.col("machine_id") == "M1004").explain()

In [0]:
vends.select("machine_id", "amount").explain()

In [0]:
vends.withColumn("net", F.col("amount") * 0.91).explain()

In [0]:
vends.groupBy("machine_id").sum("amount").explain()

In [0]:
vends.orderBy("amount").explain()

In [0]:
vends.select("category").distinct().explain()

In [0]:
vends.groupBy("machine_id", "vend_date").sum("amount") \
     .filter(F.col("vend_date") == "2026-05-15").explain()

In [0]:
vends.join(machines, "machine_id").groupBy("location_type").sum("amount").explain()
vends.join(machines.hint("merge"), "machine_id").groupBy("location_type").sum("amount").explain()

In [0]:
vends.filter(F.col("vend_date") == "2026-05-15").select("machine_id", "amount").explain("formatted")

In [0]:
vends.select("machine_id", "amount").explain("formatted")

In [0]:
vends.write.mode("overwrite").partitionBy("vend_date").saveAsTable("databricksbootcamp.northwindcasestudy.vends_by_date")
spark.table("databricksbootcamp.northwindcasestudy.vends_by_date").filter(F.col("vend_date") == "2026-05-15").explain("formatted")

In [0]:
spark.sql("DESCRIBE DETAIL databricksbootcamp.northwindcasestudy.vends").display()
spark.sql("OPTIMIZE databricksbootcamp.northwindcasestudy.vends")
spark.sql("DESCRIBE DETAIL databricksbootcamp.northwindcasestudy.vends").display()

In [0]:
vends.write.clusterBy("machine_id", "vend_date").mode("overwrite").saveAsTable("databricksbootcamp.northwindcasestudy.vends_clustered")
spark.sql("OPTIMIZE databricksbootcamp.northwindcasestudy.vends_clustered")
spark.sql("DESCRIBE DETAIL databricksbootcamp.northwindcasestudy.vends_clustered").display()

In [0]:
vends.join(machines, "machine_id").explain()
vends.join(broadcast(machines), "machine_id").explain()

In [0]:
vends.join(machines.hint("merge"), "machine_id").explain()

In [0]:
q = vends.groupBy("machine_id").sum("amount")
q.explain()
q.count()

In [0]:
vends.join(machines.hint("merge"), "machine_id") \
     .filter(F.col("vend_date").between("2026-05-01", "2026-05-31")) \
     .groupBy("location_type").sum("amount").explain()

spark.table("databricksbootcamp.northwindcasestudy.vends_clustered") \
     .filter(F.col("vend_date").between("2026-05-01", "2026-05-31")) \
     .join(broadcast(machines), "machine_id") \
     .groupBy("location_type").sum("amount").explain()